# 06 - Análise do Espaço Latente Pré-Classificador, Avaliação Hierárquica e Auditoria de Sonda Linear

Este notebook reproduz a extração de representações latentes pré-classificador
(2048-d ResNet50 e 768-d ConvNeXt), avaliação de KNN cosseno ($k=5$), Silhouette Score de alta dimensão
com auditoria de suporte de classes, correlações hierárquicas Species $\to$ Genus $\to$ Order (Tabela 6),
auditoria de sonda linear independente em treino/validação (`AD-012`), e projeção qualitativa UMAP 2D (Figura 2).

**Invariantes científicas (`AD-012`):**
- Embeddings extraídos imediatamente antes de `model.fc` via pre-hook sem redução transdutiva de dimensionalidade.
- KNN cosseno ($k=5$) com split 80/20 disjunto para garantir estritamente zero auto-vizinhança.
- Silhouette Score no espaço original de alta dimensão com auditoria de classes com $<2$ amostras.
- UMAP 2D é estritamente uma visualização qualitativa e não expressa distância filogenética ou separação formal de clusters.
- Sonda linear para teste do gargalo de classificação treinada exclusivamente em treino/validação sem vazamento do teste.
A taxonomia é procurada em `DATA_ROOT/cnn_kraken/data_taxonomy_parquet` e em caminhos legados. Para usar a saída de uma execução do notebook 05 em `results_reproduction/runs/...`, defina `TAXONOMY_PATH` como o caminho desse Parquet antes de iniciar o notebook 06. A versão `CLEANSED` tem prioridade; o de-para híbrido original também é aceito após validação dos IDs e níveis usados.

Quando um gênero ou ordem está ausente no de-para, as métricas daquele nível usam somente fragmentos com o rank conhecido; a métrica de espécie mantém todos os fragmentos. As coberturas por classe e por fragmento são registradas em `taxonomy_rank_coverage.json` e na Tabela 5. Não interprete resultados sobre subconjuntos como se tivessem o mesmo denominador.

Retomada validada: `LATENT_EXECUTION_MODE="reuse"` (padrão) ou `"rerun"`; origem opcional `LATENT_REUSE_RUN_ID`. Consulte `docs/latent_resume.md`.


In [ ]:
import os

# Retomada: reuse é o padrão; rerun recalcula em uma pasta nova.
# Overrides de ambiente têm precedência sobre estes parâmetros editáveis.
LATENT_EXECUTION_MODE = os.environ.get("LATENT_EXECUTION_MODE", "reuse")
LATENT_REUSE_RUN_ID = os.environ.get("LATENT_REUSE_RUN_ID") or None
if LATENT_EXECUTION_MODE not in ("reuse", "rerun"):
    raise ValueError("LATENT_EXECUTION_MODE must be reuse or rerun")
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
project_root = None
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        project_root = r
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break
if project_root is None:
    raise RuntimeError("Could not resolve repository root containing src/viral_cnn")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
    create_optimized_dataloader,
    free_gpu_memory,
    reserve_run_directory,
    preflight_latent_inputs,
)
from viral_cnn.data.parquet_dataset import ParquetBatchIterableDataset
from viral_cnn.models.registry import build_registered_model
from viral_cnn.eval.latent import (
    extract_latent_embeddings,
    _sha256_file,
    EmbeddingArtifact,
    save_embedding_artifact,
    evaluate_cosine_knn,
    save_knn_identity_audit,
    evaluate_silhouette_with_audit,
    calculate_taxonomic_correlations,
    LinearProbeClassifier,
    LatentMemoryBudget,
    audit_classifier_bottleneck,
    export_umap_projection,
)
from viral_cnn.latent_resume import (
    LatentResumeCache, StageResult, snapshot_probe, restore_probe, infer_native_predictions, notebook_code_sha256,
)
from viral_cnn.utils.logging_utils import log_artifact_path
from viral_cnn.taxonomy.curation import generate_synthetic_taxonomy, sanitize_hybrid_taxonomy

# 1. Inicialização do runtime e contexto de execução (AD-017)
ctx = setup_notebook_runtime()
if ctx.run_id is None:
    ctx.run_id = "c09-" + __import__("datetime").datetime.now().strftime("%Y%m%dT%H%M%S%f")
output_dir = reserve_run_directory(project_root, ctx.run_id, "06_ictv_latent_space_and_hierarchy")
ctx.output_dir = output_dir
resume = LatentResumeCache(output_dir, mode=LATENT_EXECUTION_MODE, reuse_run_id=LATENT_REUSE_RUN_ID)
# Hash only notebook source, so saving cell output does not invalidate progress.
notebook_path = project_root / "notebooks/reproduction/short/06_ictv_latent_space_and_hierarchy.ipynb"
stage_code_sha256 = notebook_code_sha256(notebook_path)
stage_environment = {"code": stage_code_sha256, "versions": ctx.library_versions,
                     "device": str(ctx.device), "seed": ctx.seed, "smoke": ctx.is_smoke_test,
                     "source_files": {str(p.relative_to(project_root)): _sha256_file(p)
                                      for p in sorted((project_root / "src/viral_cnn").rglob("*.py"))}}

# LATENT_RAM_BUDGET_MB sem override usa 30% da RAM disponível medida no início.
latent_eval_batch_size = int(os.environ.get("LATENT_EVAL_BATCH_SIZE", "256"))
latent_knn_workspace_mb = int(os.environ.get("LATENT_KNN_WORKSPACE_MB", "1024"))
latent_knn_query_batch_size = int(os.environ.get("LATENT_KNN_QUERY_BATCH_SIZE", "256"))
if latent_knn_workspace_mb < 1 or latent_knn_query_batch_size < 1:
    raise ValueError("LATENT_KNN_WORKSPACE_MB and LATENT_KNN_QUERY_BATCH_SIZE must be positive integers")
if latent_eval_batch_size < 1:
    raise ValueError("LATENT_EVAL_BATCH_SIZE must be a positive integer")
latent_spill_dir = Path(os.environ.get("LATENT_SPILL_DIR", output_dir / "spill"))
latent_memory = LatentMemoryBudget.from_environment(spill_dir=latent_spill_dir)
latent_ram_available_start = LatentMemoryBudget.available_ram_bytes()
latent_ram_budget_mb = latent_memory.budget_bytes / (1024 * 1024)
print(f"[LATENT_MEMORY] batch_size={latent_eval_batch_size}, budget_mb={latent_ram_budget_mb:.2f}, "
      f"available_at_start_mb={latent_ram_available_start / (1024 * 1024):.2f}")

print(f"[LATENT_KNN] device={ctx.device}, workspace_mb={latent_knn_workspace_mb}, query_batch_size={latent_knn_query_batch_size}")
print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


In [2]:
# 2. Carregamento do de-para ICTV/NCBI e validação dos rótulos
cleaned_tax_path = Path(ctx.topology.paths["short"]["hybrid_taxonomy_parquet"])
if not ctx.is_smoke_test and not cleaned_tax_path.is_file():
    raise FileNotFoundError(
        f"Taxonomia empírica ausente: {cleaned_tax_path}. "
        "Defina TAXONOMY_PATH com o arquivo Parquet real (ou seu diretório) "
        "antes de executar a primeira célula."
    )
if cleaned_tax_path.is_file():
    print(f"[LOG_PATH] Loading taxonomy from: {cleaned_tax_path.resolve()}")
    taxonomy_df = pd.read_parquet(cleaned_tax_path)
    required = {"id", "Genus", "Order"}
    missing = required - set(taxonomy_df.columns)
    if missing:
        raise ValueError(f"De-para taxonômico sem colunas obrigatórias: {sorted(missing)}")
    ids = pd.to_numeric(taxonomy_df["id"], errors="raise")
    if ids.isna().any() or (ids < 1).any() or (ids % 1 != 0).any() or ids.duplicated().any():
        raise ValueError("De-para taxonômico com IDs ausentes, inválidos ou duplicados")
    expected_labels = ids.astype("int64") - 1
    if "model_label" in taxonomy_df.columns:
        labels = pd.to_numeric(taxonomy_df["model_label"], errors="raise")
        if labels.isna().any() or not labels.eq(expected_labels).all():
            raise ValueError("model_label deve ser igual a id - 1; verifique o de-para")
    taxonomy_df["model_label"] = expected_labels
elif ctx.is_smoke_test:
    taxonomy_df = generate_synthetic_taxonomy(num_classes=100)

# Ausência de linhagem no fallback não pode se tornar um gênero ou ordem fictícios.
for rank in ("Genus", "Order"):
    names = taxonomy_df[rank].astype("string").str.strip()
    missing = names.isna() | names.eq("").fillna(False) | names.str.startswith("Unassigned", na=False)
    taxonomy_df[rank] = names.mask(missing, pd.NA)
rank_coverage = {
    rank: {"mapped_species": int(taxonomy_df[rank].notna().sum()),
           "unmapped_species": int(taxonomy_df[rank].isna().sum())}
    for rank in ("Genus", "Order")
}
coverage_path = output_dir / "taxonomy_rank_coverage.json"
coverage_path.write_text(json.dumps({"source": str(cleaned_tax_path.resolve()) if cleaned_tax_path.is_file() else "synthetic",
                                     "total_species": len(taxonomy_df), "ranks": rank_coverage}, indent=2) + "\n")
print(f"[LOG_PATH] Cobertura taxonômica registrada em: {coverage_path.resolve()}")
for rank, counts in rank_coverage.items():
    print(f"{rank}: {counts['mapped_species']} espécies com mapeamento; {counts['unmapped_species']} sem mapeamento")
    if not counts["mapped_species"]:
        raise ValueError(f"Nenhuma espécie possui {rank}; não é possível calcular métricas nesse nível")

print(f"Taxonomia carregada: {len(taxonomy_df)} espécies")
print(f"Genera: {taxonomy_df['Genus'].nunique()}, Orders: {taxonomy_df['Order'].nunique()}")


[LOG_PATH] Loading taxonomy from: /home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet
[LOG_PATH] Cobertura taxonômica registrada em: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c01-20261104T115416597521/06_ictv_latent_space_and_hierarchy/taxonomy_rank_coverage.json
Genus: 7746 espécies com mapeamento; 5937 sem mapeamento
Order: 4583 espécies com mapeamento; 9100 sem mapeamento
Taxonomia carregada: 13683 espécies
Genera: 2320, Orders: 68


In [ ]:
# 3. Extração por split com identidade operacional e checkpoint registrado
np.random.seed(ctx.seed)
torch.manual_seed(ctx.seed)
models_to_evaluate = [
    {"name": "resnet_clean_species", "display": "ResNet50", "dim": 2048},
    {"name": "resnet_noise_species", "display": "ResNet50_F", "dim": 2048},
    {"name": "convnext_zero_species", "display": "ConvNeXt_Z", "dim": 2048},
    {"name": "convnext_transfer_species", "display": "ConvNeXt_T", "dim": 2048},
    {"name": "convnext_noise_species", "display": "ConvNeXt_F", "dim": 2048},
]
checkpoint_paths = {m["name"]: ctx.topology.paths["checkpoints"].get(m["name"]) for m in models_to_evaluate}
if not ctx.is_smoke_test:
    for variant, checkpoint in checkpoint_paths.items():
        if not checkpoint or not Path(checkpoint).is_file():
            raise FileNotFoundError(f"Required checkpoint for '{variant}' not found at: {checkpoint}")
species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
if ctx.is_smoke_test:
    # Isolated deterministic fixtures: distinct sequence content for each split and run.
    fixture_dir = output_dir / "fixtures"
    fixture_dir.mkdir(parents=True, exist_ok=True)
    split_files = {}
    bases = np.array(list("ACGT"))
    for split_index, split in enumerate(("train", "validation", "test")):
        rng = np.random.default_rng(ctx.seed + split_index)
        sequences = ["".join(rng.choice(bases, size=512)) for _ in range(16)]
        fixture_path = fixture_dir / f"{split}.parquet"
        frame = pd.DataFrame({"sequence_padded": sequences, "label": [i % 5 for i in range(16)]})
        # Keep deterministic fixture identity across smoke restarts, without modifying a source.
        frame.to_parquet(fixture_path, index=False)
        chosen = fixture_path
        for source in resume.sources:
            reference = source / "smoke_split_files.json"
            candidate = source / "fixtures" / f"{split}.parquet"
            if reference.is_file():
                log_artifact_path("smoke split references", reference)
                candidate = Path(json.loads(reference.read_text())[split][0])
            if candidate.is_file():
                log_artifact_path("smoke fixture candidate", candidate)
                if pd.read_parquet(candidate).equals(frame):
                    chosen = candidate
                    break
        split_files[split] = [str(chosen.resolve())]
    (output_dir / "smoke_split_files.json").write_text(json.dumps(split_files, indent=2) + "\n")
else:
    split_files = {split: resolve_split_files(species_parquet_dir, split) for split in ("train", "validation", "test")}
partition_digests = {str(path): _sha256_file(path) for paths in split_files.values() for path in paths}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    left_hashes = {partition_digests[str(path)] for path in split_files[left]}
    right_hashes = {partition_digests[str(path)] for path in split_files[right]}
    if left_hashes.intersection(right_hashes):
        raise ValueError(f"Partition content identity overlaps between {left} and {right}")
if not ctx.is_smoke_test:
    # O preflight de hierarquia usa as classes com ambos os níveis conhecidos.
    # As métricas de espécie permanecem sobre todas as classes originais.
    complete_hierarchy = taxonomy_df.dropna(subset=["Genus", "Order"])
    if complete_hierarchy.empty:
        raise ValueError("Sem espécies com Genus e Order conhecidos para validar a hierarquia")
    preflight = preflight_latent_inputs(split_files, complete_hierarchy, checkpoint_paths, [m["name"] for m in models_to_evaluate])
    preflight["taxonomy_rank_coverage"] = rank_coverage
    preflight["taxonomy_labels_total"] = int(taxonomy_df["model_label"].nunique())
else:
    preflight = {"status": "SYNTHETIC_SMOKE", "splits": {k: [str(x) for x in v] for k, v in split_files.items()}}
    if any(not files for files in split_files.values()):
        raise FileNotFoundError("Smoke fixtures must provide distinct train, validation, and test partitions")

species_to_genus = {}
species_to_order = {}
genus_map = {g: i for i, g in enumerate(taxonomy_df["Genus"].dropna().unique())}
order_map = {o: i for i, o in enumerate(taxonomy_df["Order"].dropna().unique())}
taxonomy_species_labels = set(taxonomy_df["model_label"].astype(int))
for _, row in taxonomy_df.iterrows():
    label = int(row["model_label"])
    if pd.notna(row["Genus"]):
        species_to_genus[label] = genus_map[row["Genus"]]
    if pd.notna(row["Order"]):
        species_to_order[label] = order_map[row["Order"]]

embedding_artifacts = {}
embeddings_cache, labels_cache, identities_cache = {}, {}, {}
probe_instances, probe_audits = {}, []
if ctx.is_smoke_test:
    class SmokeLatentModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.projection = nn.Linear(512, 2048)
            self.fc = nn.Linear(2048, max(int(taxonomy_df["model_label"].max()) + 1, 1))
        def forward(self, tokens):
            return self.fc(torch.relu(self.projection(tokens.float())))

def embedding_manifest(model_spec, split, files, dataset, count, checkpoint):
    return {
        "verified": True, "variant": model_spec["name"], "split": split,
        "partitions": [str(Path(f).resolve()) for f in files],
        "partition_sha256": [partition_digests[str(f)] for f in files],
        "checkpoint": str(Path(checkpoint).resolve()) if checkpoint and Path(checkpoint).is_file() else "synthetic-smoke-initialization",
        "checkpoint_sha256": checkpoint_digests[model_spec["name"]],
        "extractor_code_sha256": extractor_digest,
        "class_order": sorted(int(x) for x in taxonomy_df["model_label"].unique()),
        "encoding": {"alphabet": "A/C/G/T", "channels": ["A", "C", "G", "T"], "length": 512},
        "mode": "smoke" if ctx.is_smoke_test else "empirical",
        "extractor": "viral_cnn.eval.latent.extract_latent_embeddings:v1",
        "population_available": dataset.total_samples, "extracted_count": count,
        "sampling": {"rule": "none", "seed": ctx.seed},
    }

# Immutable source identities are computed once per checkpoint, not per split.
checkpoint_digests = {name: _sha256_file(path) if path and Path(path).is_file() else "synthetic"
                      for name, path in checkpoint_paths.items()}
extractor_digest = _sha256_file(Path(__import__("viral_cnn.eval.latent", fromlist=["__file__"]).__file__))
taxonomy_digest = _sha256_file(cleaned_tax_path) if cleaned_tax_path.is_file() else "synthetic"
artifact_paths = {}

for model_index, model_spec in enumerate(models_to_evaluate):
    name = model_spec["name"]
    checkpoint = checkpoint_paths[name]
    if not ctx.is_smoke_test and not Path(checkpoint).is_file():
        raise FileNotFoundError(f"Required checkpoint for '{name}' not found at: {checkpoint}")
    # Isolate per-model randomness: skipping an earlier cached probe must not change a later fit.
    torch.manual_seed(ctx.seed + model_index)
    if ctx.is_smoke_test:
        print(f"[LOG_PATH] Smoke model initialized without checkpoint; expected checkpoint: {checkpoint}")
        model = SmokeLatentModel()
    else:
        log_artifact_path(f"checkpoint for {name}", Path(checkpoint).resolve())
        model = build_registered_model(name, checkpoint_path=checkpoint, device=ctx.device)
    model = model.to(ctx.device)
    model_spec["dim"] = int(model.fc.in_features)

    def obtain_split(split):
        files = split_files[split]
        dataset = ParquetBatchIterableDataset(
            files, batch_size=32 if ctx.is_smoke_test else latent_eval_batch_size, shuffle=False,
            split=split, include_identity=True, partition_ids=partition_digests,
        )
        expected = embedding_manifest(model_spec, split, files, dataset, dataset.total_samples, checkpoint)
        if ctx.is_smoke_test:
            expected["smoke_model_sha256"] = LinearProbeClassifier._state_hash(model)
        hit = resume.load_embedding(expected, model_spec["dim"], memory_manager=latent_memory,
                                    batch_size=latent_eval_batch_size)
        native = None
        if hit:
            artifact, cache_path = hit
        else:
            loader = create_optimized_dataloader(dataset, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
            if split == "test":
                values = extract_latent_embeddings(
                    model, loader, device=ctx.device, return_identities=True,
                    return_predictions=True, memory_manager=latent_memory)
            else:
                values = extract_latent_embeddings(
                    model, loader, device=ctx.device, return_identities=True, memory_manager=latent_memory)
            feats, labels, row_ids = values[:3]
            if not len(feats) or len(feats) != dataset.total_samples:
                raise ValueError(f"Incomplete {split} embedding extraction for {name}")
            missing_taxa = sorted(set(np.unique(labels).tolist()) - taxonomy_species_labels)
            if missing_taxa:
                raise ValueError(f"Taxonomy does not cover model label {missing_taxa[0]}")
            artifact = EmbeddingArtifact(feats, labels, row_ids, expected)
            artifact.validate()
            cache_path = save_embedding_artifact(artifact, output_dir / "features", batch_size=latent_eval_batch_size)
            log_artifact_path("saved embedding manifest", cache_path.with_suffix(".json"))
            artifact.manifest = json.loads(cache_path.with_suffix(".json").read_text())
            if split == "test":
                native = values[3]
            resume.record(f"embedding:{name}:{split}", "recalculated", cache_path,
                          sha256=artifact.manifest["artifact_sha256"])
        artifact_paths[(name, split)] = cache_path
        return artifact, native

    train_artifact, _ = obtain_split("train")
    validation_artifact, _ = obtain_split("validation")
    probe_identity = dict(stage_environment, variant=name, taxonomy=taxonomy_digest,
        train=train_artifact.manifest, validation=validation_artifact.manifest,
        epochs=2 if ctx.is_smoke_test else 10, lr=1e-3, weight_decay=1e-4,
        batch_size=latent_eval_batch_size, probe_protocol="c01_train_val_frozen_v1",
        initialization_protocol="c09_seeded_probe_init_v1")

    def fit_probe():
        torch.manual_seed(ctx.seed)  # c09_seeded_probe_init_v1
        probe = LinearProbeClassifier(in_features=model_spec["dim"],
                                      num_classes=max(int(taxonomy_df["model_label"].max()) + 1, 1))
        selection = probe.fit_select_and_seal(
            train_artifact, validation_artifact,
            epochs=2 if ctx.is_smoke_test else 10, lr=1e-3, weight_decay=1e-4,
            batch_size=latent_eval_batch_size, seed=ctx.seed, device=ctx.device,
        )
        return snapshot_probe(probe, selection)

    payload = resume.run_stage(f"probe_{name}", probe_identity, fit_probe,
        validate=lambda value: restore_probe(value, train_artifact, validation_artifact))
    probe, selection = restore_probe(payload, train_artifact, validation_artifact)
    probe_instances[name] = probe
    probe_audits.append({"model": name, "selection": selection})
    latent_memory.release(train_artifact.features, train_artifact.labels, train_artifact.row_ids,
                          validation_artifact.features, validation_artifact.labels, validation_artifact.row_ids)
    del train_artifact, validation_artifact, payload

    # Test is first opened after weights have been selected/sealed, including on restart.
    test_artifact, native_predictions = obtain_split("test")
    cache_path = artifact_paths[(name, "test")]
    model_spec["test_artifact_sha256"] = test_artifact.manifest["artifact_sha256"]
    native_identity = dict(stage_environment, test=test_artifact.manifest, batch_size=latent_eval_batch_size)

    def native_prediction_stage():
        if native_predictions is not None:
            return native_predictions
        test_dataset = ParquetBatchIterableDataset(
            split_files["test"], batch_size=32 if ctx.is_smoke_test else latent_eval_batch_size,
            shuffle=False, split="test", include_identity=True, partition_ids=partition_digests)
        test_loader = create_optimized_dataloader(test_dataset, is_smoke_test=ctx.is_smoke_test, device=ctx.device)
        return infer_native_predictions(model, test_loader, test_artifact, device=ctx.device)

    def validate_native(value):
        if (not isinstance(value, np.ndarray) or value.shape != test_artifact.labels.shape
                or value.dtype.kind not in "iu" or not np.isin(value, sorted(taxonomy_species_labels)).all()):
            raise ValueError("Native prediction checkpoint has invalid shape/classes")

    predictions = resume.run_stage(f"native_{name}", native_identity, native_prediction_stage, validate=validate_native)
    if native_predictions is not None and predictions is not native_predictions:
        latent_memory.release(native_predictions)
    embedding_artifacts[(name, "test")] = test_artifact
    test_feats, test_labels, test_ids = test_artifact.features, test_artifact.labels, test_artifact.row_ids
    embeddings_cache[name], labels_cache[name], identities_cache[name] = test_feats, test_labels, test_ids
    model_spec["native_test_predictions"] = predictions
    probe.to("cpu")
    free_gpu_memory(model)
    del model

first_model_name = models_to_evaluate[0]["name"]
print(f"Representações latentes prontas: 5 modelos, {len(embeddings_cache[first_model_name])} linhas de teste")


In [ ]:
# 4. Avaliação de KNN Cosseno (k=5) e Silhouette Score (Tabela 5)
table5_rows = []
per_taxon_metrics_cache = {}
knn_audits = []
silhouette_sample_sizes = {}
silhouette_retained_classes = {}
silhouette_excluded_classes = {}

for m in models_to_evaluate:
    name = m["name"]
    raw_feats = embeddings_cache[name]
    y_sp = labels_cache[name]
    per_taxon_metrics_cache[name] = {}

    for rank_name, rank_mapping in [("Species", None), ("Genus", species_to_genus), ("Order", species_to_order)]:
        eligible = (np.ones(len(y_sp), dtype=bool) if rank_mapping is None else
                    np.fromiter((int(sp) in rank_mapping for sp in y_sp), dtype=bool, count=len(y_sp)))
        rank_indices = np.flatnonzero(eligible)
        labels = y_sp[rank_indices] if rank_mapping is None else np.array(
            [rank_mapping[int(sp)] for sp in y_sp[rank_indices]], dtype=np.int64)
        rank_identity = dict(stage_environment, embedding=m["test_artifact_sha256"],
            taxonomy=taxonomy_digest, rank=rank_name, k=5, random_state=42,
            workspace_mb=latent_knn_workspace_mb, query_batch_size=latent_knn_query_batch_size)

        def compute_knn():
            result = evaluate_cosine_knn(
                features=raw_feats,
                labels=labels,
                k=5,
                random_state=42,
                taxonomic_rank=rank_name,
                row_ids=identities_cache[name],
                selected_indices=rank_indices,
                workspace_mb=latent_knn_workspace_mb,
                query_batch_size=latent_knn_query_batch_size,
                device=ctx.device,
                compact=True,
                audit_output_path=output_dir / f"knn_neighbors_{name}_{rank_name.lower()}.parquet",
                embedding_artifact_sha256=m["test_artifact_sha256"],
            )
            # Query arrays are not needed downstream; retain only the compact summary/frame.
            for key in ("reference_indices", "query_indices", "y_true", "y_pred"):
                result.pop(key, None)
            audit_path = result["audit_summary"].get("neighbor_parquet_path")
            return StageResult(result, [audit_path] if audit_path else [])

        knn_res = resume.run_stage(f"knn_{name}_{rank_name.lower()}", rank_identity, compute_knn)
        knn_audits.append(dict(knn_res["audit_summary"], model=name))

        def compute_silhouette():
            return evaluate_silhouette_with_audit(
                features=raw_feats,
                labels=labels,
                selected_indices=rank_indices,
                max_samples=min(len(rank_indices), 200 if ctx.is_smoke_test else 5000),
                random_state=42,
                taxonomic_rank=rank_name,
            )

        sil_identity = dict(stage_environment, embedding=m["test_artifact_sha256"], taxonomy=taxonomy_digest,
                            rank=rank_name, random_state=42,
                            max_samples=min(len(rank_indices), 200 if ctx.is_smoke_test else 5000))
        sil_res = resume.run_stage(f"silhouette_{name}_{rank_name.lower()}", sil_identity, compute_silhouette)
        silhouette_sample_sizes[f"{name}:{rank_name}"] = sil_res["actual_sample_size"]
        silhouette_retained_classes[f"{name}:{rank_name}"] = sil_res["valid_classes_count"]
        silhouette_excluded_classes[f"{name}:{rank_name}"] = sil_res["excluded_classes_count"]
        if not knn_res["available"]:
            knn_per_cluster = pd.DataFrame(columns=["class_label", "knn_accuracy", "knn_top5_accuracy"])
        else:
            knn_per_cluster = knn_res["per_cluster_metrics"]
        sil_per_cluster = sil_res["per_cluster_metrics"]
        merged_cluster = knn_per_cluster.merge(
            sil_per_cluster[["class_label", "silhouette_score"]],
            on="class_label",
            how="left",
        )
        per_taxon_metrics_cache[name][rank_name] = merged_cluster
        table5_rows.append({
            "Model": m["display"],
            "Rank": rank_name,
            "Taxonomy_Eligible_Rows": int(eligible.sum()),
            "Taxonomy_Excluded_Rows": int((~eligible).sum()),
            "KNN_Top1": knn_res.get("global_knn_accuracy", np.nan),
            "KNN_Top5": knn_res["global_knn_top5_accuracy"],
            "Silhouette_Score": sil_res["global_silhouette_score"],
            "Silhouette_Valid_Class_Rate": sil_res.get("valid_class_rate", 0.0),
            "Silhouette_Retained_Sample_Rate": sil_res.get("retained_sample_rate", 1.0),
            "KNN_Available": knn_res["available"],
            "KNN_Queries": knn_res["n_query"],
            "KNN_Reference": knn_res["n_reference"],
            "KNN_Reference_Query_ID_Overlap": knn_res["identity_overlap_count"],
            "KNN_Unavailable_Reason": knn_res.get("unavailable_reason"),
            "KNN_Valid_Class_Count": knn_res.get("n_classes_with_query", 0),
            "KNN_Top5_Definition": knn_res["top5_definition"],
            "Silhouette_Actual_Sample_Size": sil_res["actual_sample_size"],
            "Silhouette_Total_Classes": sil_res["total_classes_sampled"],
            "Silhouette_Valid_Classes": sil_res["valid_classes_count"],
            "Silhouette_Excluded_Classes": sil_res["excluded_classes_count"],
            "Silhouette_Sampling_Strategy": sil_res["sampling_strategy"],
        })

table5_df = pd.DataFrame(table5_rows)
table5_path = output_dir / "table5_latent_clustering_metrics.csv"
table5_df.to_csv(table5_path, index=False)
knn_audit_path = output_dir / "knn_identity_audit.json"
save_knn_identity_audit(knn_audits, knn_audit_path)
print("Tabela 5 - Métricas de Espaço Latente por Nível Taxonômico:")
print(table5_df.to_string(index=False))


In [ ]:
# 5. Correlações Hierárquicas: Species -> Genus e Genus -> Order (Tabela 6)
table6_rows = []

for m in models_to_evaluate:
    m_metrics = per_taxon_metrics_cache[m["name"]]
    df_sp = m_metrics["Species"].copy()
    df_ge = m_metrics["Genus"].copy()
    df_ord = m_metrics["Order"].copy()

    mapped_species = sorted(taxonomy_species_labels)
    tax_mapping_df = pd.DataFrame({
        "species": mapped_species,
        "Genus": [species_to_genus.get(k, np.nan) for k in mapped_species],
        "Order": [species_to_order.get(k, np.nan) for k in mapped_species],
    })

    sp_genus_corr = calculate_taxonomic_correlations(
        child_df=df_sp,
        parent_df=df_ge,
        taxonomy_df=tax_mapping_df,
        child_tax_col="species",
        parent_tax_col="Genus",
        child_level="Species",
        parent_level="Genus",
        min_children=1 if ctx.is_smoke_test else 3,
    )
    table6_rows.append({
        "Model": m["display"],
        "Transition": "Species -> Genus",
        "Pearson_r": sp_genus_corr["pearson_r_knn"],
        "Spearman_rho": sp_genus_corr["spearman_rho_knn"],
        "N_taxa": sp_genus_corr["n_parents_knn"],
    })

    genus_order_corr = calculate_taxonomic_correlations(
        child_df=df_ge,
        parent_df=df_ord,
        taxonomy_df=tax_mapping_df,
        child_tax_col="Genus",
        parent_tax_col="Order",
        child_level="Genus",
        parent_level="Order",
        min_children=1 if ctx.is_smoke_test else 3,
    )
    table6_rows.append({
        "Model": m["display"],
        "Transition": "Genus -> Order",
        "Pearson_r": genus_order_corr["pearson_r_knn"],
        "Spearman_rho": genus_order_corr["spearman_rho_knn"],
        "N_taxa": genus_order_corr["n_parents_knn"],
    })

table6_df = pd.DataFrame(table6_rows)
table6_path = output_dir / "table6_taxonomic_rank_correlations.csv"
table6_df.to_csv(table6_path, index=False)
print("")
print("Tabela 6 - Correlações Hierárquicas entre Níveis Taxonômicos:")
print(table6_df.to_string(index=False))


In [ ]:
# 6. Auditoria da sonda selada no conjunto test (AD-012)
probe_results = []
for model_spec in models_to_evaluate:
    name = model_spec["name"]
    probe = probe_instances[name]
    test = embedding_artifacts[(name, "test")]
    evaluation_identity = dict(stage_environment, test=test.manifest,
        weights_sha256=probe._sealed_sha256, selection=probe_audits[len(probe_results)]["selection"],
        batch_size=latent_eval_batch_size)
    eval_res = resume.run_stage(f"probe_test_{name}", evaluation_identity,
        lambda: probe.evaluate_artifact(test, device=ctx.device, batch_size=latent_eval_batch_size, include_row_ids=False))
    native_pred = model_spec["native_test_predictions"]
    native_model_accuracy = float(np.mean(native_pred == test.labels))
    probe_results.append({
        "Model": model_spec["display"], "Linear_Probe_Test_Acc": eval_res["top1_accuracy"],
        "Linear_Probe_Macro_F1": eval_res["macro_f1"], "N": eval_res["n_test_samples"],
        "Supported_Classes": eval_res["supported_classes"], "Test_Weights_SHA256": eval_res["weights_sha256"],
        "Test_Row_IDs_SHA256": eval_res["row_ids_sha256"], "Native_Head_Accuracy": native_model_accuracy,
        "Native_Head_Comparison": "same identified test examples",
    })
probe_df = pd.DataFrame(probe_results)
probe_path = output_dir / "linear_probe_audit.csv"
probe_df.to_csv(probe_path, index=False)
(output_dir / "linear_probe_selection.json").write_text(json.dumps(probe_audits, indent=2, default=str) + "\n")
print(probe_df.drop(columns=["Test_Row_IDs_SHA256"]).to_string(index=False))


In [ ]:
# 7. Projeção Qualitativa UMAP 2D (Figura 2)
umap_feats = embeddings_cache["convnext_noise_species"]
umap_y = labels_cache["convnext_noise_species"]
umap_sample_size = min(100, len(umap_feats))
def compute_umap():
    projection = export_umap_projection(
        features=umap_feats[:100],
        labels=umap_y[:100],
        output_path=output_dir / "umap_projection_convnext_f.csv",
        random_state=42,
    )
    return StageResult(projection, [output_dir / "umap_projection_convnext_f.csv"])

umap_df = resume.run_stage("umap", dict(stage_environment,
    embedding=models_to_evaluate[-1]["test_artifact_sha256"], random_state=42, sample_size=umap_sample_size), compute_umap)
if not (output_dir / "umap_projection_convnext_f.csv").exists():
    umap_df.to_csv(output_dir / "umap_projection_convnext_f.csv", index=False)
print("")
print("[UMAP] Projeção UMAP 2D gerada com sucesso.")
print("AVISO CIENTÍFICO (AD-012): A projeção UMAP 2D destina-se estritamente à visualização qualitativa")
print("e NÃO demonstra distância filogenética ou separação formal de clusters.")


In [ ]:
# 8. Emissão do Manifesto de Execução Rastreável
manifest = write_notebook_manifest(
    notebook_id="06_ictv_latent_space_and_hierarchy",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    output_json_path=output_dir / "manifest.json",
    run_id=ctx.run_id,
    input_files=[str(cleaned_tax_path)] +
        list(dict.fromkeys(e["path"] for e in resume.events if e["status"] == "reused" and "path" in e)) + [str(path) for paths in split_files.values() for path in paths],
    checkpoints=checkpoint_paths,
    outputs=[str(path) for path in output_dir.iterdir() if path.is_file() and path.name != "manifest.json"],
    configuration={"latent_execution_mode": resume.mode, "latent_reuse_sources": [str(p) for p in resume.sources], "resume_report": str(output_dir / "resume_report.json"), "probe_initialization": "c09_seeded_probe_init_v1", "probe_protocol": "c01_train_val_frozen_v1", "probe_epochs": 2 if ctx.is_smoke_test else 10, "probe_lr": 1e-3, "probe_weight_decay": 1e-4, "probe_batch_size": latent_eval_batch_size, "latent_eval_batch_size": latent_eval_batch_size, "latent_ram_budget_mb": latent_ram_budget_mb, "latent_ram_available_start_mb": latent_ram_available_start / (1024 * 1024), "latent_spill_dir": str(latent_spill_dir.resolve()), "latent_storage": latent_memory.report(), "probe_seed": ctx.seed, "knn_k": 5, "knn_test_reference_fraction": 0.8, "knn_protocol": "c08_tiled_cosine_v1", "latent_knn_workspace_mb": latent_knn_workspace_mb, "latent_knn_query_batch_size": latent_knn_query_batch_size, "latent_knn_device": str(ctx.device), "knn_effective_blocks": {f"{audit['model']}:{audit['rank']}": {"query": audit["query_batch_size_effective"], "reference": audit["reference_batch_size_effective"]} for audit in knn_audits}, "silhouette_sample_sizes": silhouette_sample_sizes, "silhouette_retained_classes": silhouette_retained_classes, "silhouette_excluded_classes": silhouette_excluded_classes, "umap_sample_size": umap_sample_size, "genome_independence": "unknown"},
    metrics_summary={
        "models_evaluated": len(models_to_evaluate),
        "table5_entries": len(table5_df),
        "table6_entries": len(table6_df),
        "umap_exported": True,
        "probe_variants": len(probe_df),
        "preflight": preflight,
        "genome_independence": "unknown: genome IDs unavailable for test fragments",
    },
)
latent_memory.close()
print("\nManifesto de Execução salvo com sucesso:")
print(json.dumps(manifest.metrics_summary, indent=2))
